# Perplexity Evaluation

This notebook evaluates how well the current N-gram and RNN models fit Indonesian test text from `model/specil_test.csv`.

Lower perplexity means the model is less confused by the held-out corpus.

In [7]:
from pathlib import Path
import csv
import math
import sys
import tensorflow as tf
ROOT = Path.cwd()
if not (ROOT / 'app.py').exists():
    ROOT = ROOT.parent.parent

sys.path.insert(0, str(ROOT / 'model' / 'n_gram'))
sys.path.insert(0, str(ROOT / 'model' / 'RNN'))

from ngram_spell_checker import NGramSpellChecker, tokenize_words
try:
    from specil_rnn_spellchecker import SpecilRnnSpellChecker
    RNN_AVAILABLE = True
except ModuleNotFoundError as e:
    SpecilRnnSpellChecker = None
    RNN_AVAILABLE = False
    print('RNN skipped:', e)
    print('Error in importing library')

TEST_PATH = ROOT / 'model' / 'specil_test.csv'
MAX_ROWS = 1000
print(ROOT)


e:\8. Mata Kuliah\Semester 6\IoT Kecerdasan Artifisial\FinalProject_MorseNLP


In [8]:
from pathlib import Path
import csv
import math
import sys

ROOT = Path.cwd()
if not (ROOT / 'app.py').exists():
    ROOT = ROOT.parent.parent

sys.path.insert(0, str(ROOT / 'model' / 'n_gram'))
sys.path.insert(0, str(ROOT / 'model' / 'RNN'))

from ngram_spell_checker import NGramSpellChecker, tokenize_words
from specil_rnn_spellchecker import SpecilRnnSpellChecker

TEST_PATH = ROOT / 'model' / 'specil_test.csv'
MAX_ROWS = 1000
print(ROOT)

e:\8. Mata Kuliah\Semester 6\IoT Kecerdasan Artifisial\FinalProject_MorseNLP


In [9]:
def load_sentences(limit=MAX_ROWS):
    rows = []
    with TEST_PATH.open(encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            rows.append(row['correct_text'])
            if len(rows) >= limit:
                break
    return rows

sentences = load_sentences()
len(sentences), sentences[:2]

(1000,
 ['diskusikan gambar sampul di atas dengan menjawab pertanyaan-pertanyaan ini .',
  'bola boni biru .'])

## N-gram Perplexity

The current N-gram model is a character N-gram spell checker, not a word next-token language model. For a practical corpus-fit proxy, this computes average token negative log probability from its learned word-frequency table.

In [10]:
if RNN_AVAILABLE:
    rnn = SpecilRnnSpellChecker.load()

    def rnn_perplexity(sentences):
        scores = [rnn.score(sentence) for sentence in sentences]
        return math.exp(-sum(scores) / max(len(scores), 1))

    rnn_ppl = rnn_perplexity(sentences[:100])
else:
    rnn_ppl = None
rnn_ppl


9.8803029270859

print({'ngram_perplexity': ngram_ppl, 'rnn_perplexity': rnn_ppl})

In [ ]:
rnn = SpecilRnnSpellChecker.load()

def rnn_perplexity(sentences):
    scores = [rnn.score(sentence) for sentence in sentences]
    return math.exp(-sum(scores) / max(len(scores), 1))

rnn_ppl = rnn_perplexity(sentences[:100])
rnn_ppl

In [ ]:
print({'ngram_perplexity': ngram_ppl, 'rnn_perplexity': rnn_ppl})

NameError: name 'ngram_ppl' is not defined

## Matplotlib Perplexity Visualization

Lower perplexity is better. Install matplotlib with `.env\python.exe -m pip install matplotlib` before running this cell.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError as e:
    raise ModuleNotFoundError('matplotlib is required. Run: .env\\python.exe -m pip install matplotlib') from e

labels = ['N-gram']
values = [ngram_ppl]
if rnn_ppl is not None:
    labels.append('RNN')
    values.append(rnn_ppl)

plt.figure(figsize=(7, 4))
plt.bar(labels, values, color=['#60a5fa', '#f87171'][:len(labels)])
plt.title('Model Perplexity: lower is better')
plt.ylabel('Perplexity')
for i, value in enumerate(values):
    plt.text(i, value, f'{value:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()
